# GridGuard AI - Sprint 2: Feature Engineering & Baseline Failure Probability Regressors
**Track:** Student 3: AI Engineer 1 (Tabular ML & Grid Diagnostics)
**Lane:** Lane 2: AI Service (Tabular ML)
**Brief:** Implement Feature Engineering pipeline for smart grid assets (kVA capacity, thermal loading, weather encoding) and train Baseline Failure Probability regressors (Ridge, Random Forest).

---
### Core Deliverables in this Notebook:
1. **Interaction Features Engineering:** Computing `Load_Ratio`, `Thermal_Stress`, and `Weather_Risk_Weight`.
2. **Strict Pipeline Immutability:** Preventing data leakage across train and holdout validation sets.
3. **Zero-Baseline Verification:** Documenting raw telemetry failure prediction baseline ($R^2 \approx 0.00$).
4. **Baseline Continuous Regressors:** Training Ridge Regression and Random Forest Regressor ($R^2 > 0.92$).
5. **Comparative Evaluation Harness:** Computing $R^2$, MAE, and RMSE.
6. **Artifact Serialization:** Exporting feature transformers and trained models via Joblib.


In [1]:
# 1. Environment & Library Setup
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Add ai-service to sys.path
project_root = Path('..').resolve()
ai_service_path = project_root / 'ai-service'
if str(ai_service_path) not in sys.path:
    sys.path.insert(0, str(ai_service_path))

from src.features.feature_builder import prepare_training_data, build_feature_pipeline, get_feature_column_specs
from src.models.baseline_regressors import train_baseline_suite, load_model_artifacts
from src.evaluation.metrics import evaluate_model_holdout, format_metrics_comparison_table

print('Libraries and custom modules successfully imported!')


Libraries and custom modules successfully imported!


## 2. Data Loading & Feature Inspection
We load the curated smart grid asset management records (50,000 rows) with pre-incident features and target `Failure_Probability`.


In [2]:
X_train, X_test, y_train, y_test = prepare_training_data(test_size=0.2, random_state=42)
print(f'Training Features Shape: {X_train.shape}, Target Shape: {y_train.shape}')
print(f'Holdout Test Features Shape: {X_test.shape}, Target Shape: {y_test.shape}')
print('\nTarget Summary Statistics (Failure_Probability):')
print(y_train.describe().round(4))


Training Features Shape: (40000, 21), Target Shape: (40000,)
Holdout Test Features Shape: (10000, 21), Target Shape: (10000,)

Target Summary Statistics (Failure_Probability):
count    40000.0000
mean         0.3705
std          0.1699
min          0.0500
25%          0.2300
50%          0.3600
75%          0.5000
max          0.8500
Name: Failure_Probability, dtype: float64


## 3. Domain Interaction Feature Engineering
We implement domain-specific features reflecting electrical grid physics:
- $\text{Load\_Ratio} = \frac{\text{Load\_Percentage}}{\text{Capacity\_kVA}}$: captures asset strain relative to rated nameplate capacity.
- $\text{Thermal\_Stress} = \frac{\text{Temperature}}{\text{Ambient\_Max}}$: captures thermal over-heating beyond rated ambient limits.
- $\text{Weather\_Risk\_Weight}$: encodes weather environmental hazards (Storms=3.5, Rain=2.0, Heatwave=2.8, Sunny=1.0).


In [3]:
from src.features.transformers import InteractionFeaturesTransformer

transformer = InteractionFeaturesTransformer()
sample_transformed = transformer.fit_transform(X_train.head(5))

interaction_cols = ['Load_Percentage', 'Capacity_kVA', 'Load_Ratio', 'Temperature', 'Thermal_Stress', 'Weather_Condition', 'Weather_Risk_Weight']
display_cols = [c for c in interaction_cols if c in sample_transformed.columns]
sample_transformed[display_cols].round(4)


   Load_Percentage  Capacity_kVA  Load_Ratio  Temperature  Thermal_Stress Weather_Condition  Weather_Risk_Weight
0             67.0          50.0      1.3400         48.0            0.96             Rainy                  2.0
1             78.0         500.0      0.1560         60.0            1.20             Sunny                  1.0
2             45.0         100.0      0.4500         35.0            0.70            Cloudy                  1.2
3             92.0          25.0      3.6800         75.0            1.50          Heatwave                  2.8
4             88.0         200.0      0.4400         52.0            1.04             Storm                  3.5


## 4. Model Training & Evaluation Suite
We train and benchmark three distinct regression pipelines:
1. **Raw Telemetry Baseline:** Ridge regression on raw numerical features alone without `Health_Score` (to verify $R^2 \approx 0.00$).
2. **Ridge Regressor with Engineered Pipeline:** Regularized linear model incorporating interaction features and health score.
3. **Random Forest Regressor:** Non-linear tree ensemble capturing complex equipment degradation interactions.


In [4]:
print('Training baseline models suite and calculating holdout test metrics...')
results = train_baseline_suite(X_train, y_train, X_test, y_test)
comparison_table = results['comparison_table']
print('\n=== SPRINT 2 REGRESSION PERFORMANCE BENCHMARK ===')
print(comparison_table.to_string(index=False))


Training baseline models suite and calculating holdout test metrics...

=== SPRINT 2 REGRESSION PERFORMANCE BENCHMARK ===
                                   model_name  r2_score    mae   rmse  max_error  explained_variance  sample_count
     Raw Telemetry Baseline (No Health_Score)   -0.0005 0.1441 0.1699     0.4237             -0.0005         10000
        Ridge Regressor (Engineered Features)    0.9228 0.0354 0.0472     0.1443              0.9228         10000
Random Forest Regressor (Engineered Features)    0.9214 0.0355 0.0476     0.1612              0.9214         10000


## 5. Visual Diagnostics & Error Analysis
Visualizing actual vs. predicted values, error residual distributions, and top feature importance rankings.


In [5]:
# Display generated high-resolution analytical plots
from IPython.display import Image, display

reports_dir = project_root / 'ai-service' / 'data' / 'reports'
p1 = reports_dir / 'actual_vs_predicted_regression.png'
p2 = reports_dir / 'residual_distribution_comparison.png'
p3 = reports_dir / 'feature_importance_ranking.png'

if p1.exists():
    display(Image(filename=str(p1)))
if p2.exists():
    display(Image(filename=str(p2)))
if p3.exists():
    display(Image(filename=str(p3)))


## 6. Joblib Model Serialization & Artifacts Verification
Verifying that trained feature transformers and models are persisted correctly to disk and reloadable.


In [6]:
artifacts = load_model_artifacts()
print('Successfully verified and loaded artifacts from disk:')
for name, obj in artifacts.items():
    print(f' - {name}: {type(obj).__name__}')

# Quick sanity prediction test
sample_pred = artifacts['rf_baseline'].predict(X_test.head(3))
print(f'\nSample Predictions (Failure_Probability): {sample_pred.round(4)}')
print(f'Actual Values: {y_test.head(3).values.round(4)}')


Successfully verified and loaded artifacts from disk:
 - feature_pipeline: Pipeline
 - ridge_baseline: Pipeline
 - rf_baseline: Pipeline
 - raw_telemetry_baseline: Pipeline

Sample Predictions (Failure_Probability): [0.3475 0.2688 0.4982]
Actual Values: [0.35 0.27 0.5 ]
